<a href="https://colab.research.google.com/github/LCaravaggio/Happiness_Polarization/blob/main/GDELT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import requests
import zipfile
import io
from datetime import datetime, timedelta

# ============================================================
# CONFIGURACIÓN: MODIFICAR ESTOS VALORES
# ============================================================

pais = "CO"                  # Código GDELT: CO, BR, MX, AR, etc.
fecha = "2018-06-17"         # Formato YYYY-MM-DD
dias_ventana = 2             # Días adicionales antes y después

# ============================================================
# DESCARGA DE ARCHIVOS GDELT EVENTS 1.0
# ============================================================

fecha_dt = datetime.strptime(fecha, "%Y-%m-%d")
registros = []

for desplazamiento in range(-dias_ventana, dias_ventana + 1):

    fecha_archivo = fecha_dt + timedelta(days=desplazamiento)
    fecha_str = fecha_archivo.strftime("%Y%m%d")

    url = (
        f"https://data.gdeltproject.org/events/"
        f"{fecha_str}.export.CSV.zip"
    )

    try:
        response = requests.get(url, timeout=120)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            archivo = z.namelist()[0]

            df_temp = pd.read_csv(
                z.open(archivo),
                sep="\t",
                header=None,
                usecols=[1, 29, 33, 51],
                dtype={1: int, 29: int, 51: str},
                low_memory=False
            )

        # Filtrar por país y fecha del acontecimiento
        df_temp = df_temp[
            (df_temp[51] == pais) &
            (df_temp[1] == int(fecha_dt.strftime("%Y%m%d")))
        ]

        registros.append(df_temp)

    except requests.exceptions.HTTPError as e:
        if response.status_code == 404:
            print(f"Archivo no disponible: {fecha_str}")
        else:
            raise e

# ============================================================
# CÁLCULO DEL INDICADOR
# ============================================================

if not registros:
    raise ValueError("No se encontraron registros para el país y fecha.")

df = pd.concat(registros, ignore_index=True)

# Eliminar eventuales duplicados
df = df.drop_duplicates()

# Artículos asociados a acontecimientos conflictivos
articulos_conflicto = df.loc[df[29].isin([3, 4]), 33].sum()

# Total de artículos asociados a acontecimientos
articulos_totales = df[33].sum()

# Indicador
indicador = (
    100 * articulos_conflicto / articulos_totales
    if articulos_totales > 0 else float("nan")
)

# ============================================================
# RESULTADOS
# ============================================================

print(f"País: {pais}")
print(f"Fecha: {fecha}")
print(f"Registros: {len(df)}")
print(f"Artículos totales: {articulos_totales}")
print(f"Artículos asociados a conflictos: {articulos_conflicto}")
print(f"\nProporción de cobertura de acontecimientos conflictivos: "
      f"{indicador:.2f}%")

País: CO
Fecha: 2018-06-18
Registros: 119
Artículos totales: 15674
Artículos asociados a conflictos: 4598

Proporción de cobertura de acontecimientos conflictivos: 29.34%


In [10]:
import pandas as pd
import requests
import zipfile
import io
from datetime import timedelta

# ============================================================
# CONFIGURACIÓN
# ============================================================

pais = "CO"
fecha_evento = "2018-06-17"

dias_basales = 30
dias_posteriores = 7

# Días adicionales de descarga para capturar registros
# incorporados después de la fecha del acontecimiento
dias_adicionales_descarga = 2

# ============================================================
# DEFINIR PERÍODOS
# ============================================================

evento = pd.Timestamp(fecha_evento)

inicio_basal = evento - pd.Timedelta(days=dias_basales)
fin_basal = evento - pd.Timedelta(days=1)

inicio_post = evento + pd.Timedelta(days=1)
fin_post = evento + pd.Timedelta(days=dias_posteriores)

inicio_descarga = inicio_basal
fin_descarga = fin_post + pd.Timedelta(
    days=dias_adicionales_descarga
)

print("Período basal:", inicio_basal.date(), "-", fin_basal.date())
print("Día del evento:", evento.date())
print("Período posterior:", inicio_post.date(), "-", fin_post.date())

# ============================================================
# DESCARGAR GDELT EVENTS
# ============================================================

registros = []
session = requests.Session()

fechas_archivos = pd.date_range(
    inicio_descarga,
    fin_descarga,
    freq="D"
)

for fecha_archivo in fechas_archivos:

    fecha_str = fecha_archivo.strftime("%Y%m%d")

    url = (
        f"https://data.gdeltproject.org/events/"
        f"{fecha_str}.export.CSV.zip"
    )

    try:
        response = session.get(url, timeout=120)

        if response.status_code == 404:
            print("Archivo no disponible:", fecha_str)
            continue

        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as z:

            archivo = z.namelist()[0]

            df_temp = pd.read_csv(
                z.open(archivo),
                sep="\t",
                header=None,
                usecols=[0, 1, 29, 33, 51],
                dtype={
                    0: str,
                    1: str,
                    29: "Int64",
                    51: str
                },
                low_memory=False
            )

        # Filtrar por país
        df_temp = df_temp[df_temp[51] == pais].copy()

        registros.append(df_temp)

        print(f"Descargado: {fecha_str}")

    except Exception as e:
        print(f"Error en {fecha_str}: {e}")

# ============================================================
# UNIFICAR Y FILTRAR FECHAS DE ACONTECIMIENTOS
# ============================================================

if not registros:
    raise ValueError("No se descargaron registros.")

df = pd.concat(registros, ignore_index=True)

df.columns = [
    "GlobalEventID",
    "SQLDATE",
    "QuadClass",
    "NumArticles",
    "ActionGeo_CountryCode"
]

df["fecha"] = pd.to_datetime(
    df["SQLDATE"],
    format="%Y%m%d",
    errors="coerce"
)

df["NumArticles"] = pd.to_numeric(
    df["NumArticles"],
    errors="coerce"
).fillna(0)

# Conservar acontecimientos de las fechas de análisis
df = df[
    (df["fecha"] >= inicio_basal) &
    (df["fecha"] <= fin_post)
].copy()

# Eliminar duplicados
df = df.drop_duplicates(subset="GlobalEventID")

# ============================================================
# INDICADOR DIARIO
# ============================================================

df["articulos_conflicto"] = df["NumArticles"].where(
    df["QuadClass"].isin([3, 4]), 0
)

diario = (
    df.groupby("fecha")
    .agg(
        articulos_totales=("NumArticles", "sum"),
        articulos_conflicto=("articulos_conflicto", "sum")
    )
)

# Incorporar todos los días, incluidos aquellos sin registros
calendario = pd.date_range(
    inicio_basal,
    fin_post,
    freq="D"
)

diario = diario.reindex(calendario)
diario.index.name = "fecha"

# Indicador expresado en porcentaje
diario["indicador"] = (
    100 * diario["articulos_conflicto"]
    / diario["articulos_totales"]
)

# ============================================================
# COMPARACIÓN ENTRE PERÍODOS
# ============================================================

basal = diario.loc[inicio_basal:fin_basal, "indicador"].dropna()

dia_evento = diario.loc[evento, "indicador"]

posterior = diario.loc[inicio_post:fin_post, "indicador"].dropna()

media_basal = basal.mean()
sd_basal = basal.std()

media_posterior = posterior.mean()

# Diferencias
salto_evento = dia_evento - media_basal
salto_posterior = media_posterior - media_basal

# Diferencias relativas
salto_rel_evento = (
    100 * salto_evento / media_basal
    if media_basal != 0 else float("nan")
)

salto_rel_posterior = (
    100 * salto_posterior / media_basal
    if media_basal != 0 else float("nan")
)

# Valores estandarizados
z_evento = (
    salto_evento / sd_basal
    if sd_basal > 0 else float("nan")
)

z_posterior = (
    salto_posterior / sd_basal
    if sd_basal > 0 else float("nan")
)

# ============================================================
# RESULTADOS
# ============================================================

print("\n" + "=" * 55)
print("RESULTADOS: CONFLICTIVIDAD MEDIÁTICA")
print("=" * 55)

print(f"\nPaís: {pais}")
print(f"Evento: {fecha_evento}")

print("\nCOBERTURA DEL PERÍODO:")
print(f"Días basales con datos: {len(basal)} de {dias_basales}")
print(f"Día del evento con datos: {pd.notna(dia_evento)}")
print(f"Días posteriores con datos: {len(posterior)} de {dias_posteriores}")

print("\nINDICADOR (%):")
print(f"Promedio basal:       {media_basal:.2f}")
print(f"Día del evento:       {dia_evento:.2f}")
print(f"Promedio posterior:   {media_posterior:.2f}")

print("\nDIFERENCIA RESPECTO DEL VALOR BASAL:")
print(f"Salto día del evento: {salto_evento:+.2f} puntos porcentuales")
print(f"Salto posterior:      {salto_posterior:+.2f} puntos porcentuales")

print("\nVARIACIÓN RELATIVA:")
print(f"Día del evento: {salto_rel_evento:+.2f}%")
print(f"Posterior:      {salto_rel_posterior:+.2f}%")

print("\nVALORES ESTANDARIZADOS (Z):")
print(f"Día del evento: {z_evento:+.2f}")
print(f"Posterior:      {z_posterior:+.2f}")

print("\nINDICADOR DIARIO:")
print(diario.round(2).to_string())

Período basal: 2018-05-18 - 2018-06-16
Día del evento: 2018-06-17
Período posterior: 2018-06-18 - 2018-06-24
Descargado: 20180518
Descargado: 20180519
Descargado: 20180520
Descargado: 20180521
Descargado: 20180522
Descargado: 20180523
Descargado: 20180524
Descargado: 20180525
Descargado: 20180526
Descargado: 20180527
Descargado: 20180528
Descargado: 20180529
Descargado: 20180530
Descargado: 20180531
Descargado: 20180601
Descargado: 20180602
Descargado: 20180603
Descargado: 20180604
Descargado: 20180605
Descargado: 20180606
Descargado: 20180607
Descargado: 20180608
Descargado: 20180609
Descargado: 20180610
Descargado: 20180611
Descargado: 20180612
Descargado: 20180613
Descargado: 20180614
Descargado: 20180615
Descargado: 20180616
Descargado: 20180617
Descargado: 20180618
Descargado: 20180619
Descargado: 20180620
Descargado: 20180621
Descargado: 20180622
Descargado: 20180623
Descargado: 20180624
Descargado: 20180625
Descargado: 20180626

RESULTADOS: CONFLICTIVIDAD MEDIÁTICA

País: CO
Eve

In [11]:
import pandas as pd
import numpy as np
import requests
import zipfile
import time
from pathlib import Path

# ============================================================
# 1. EVENTOS
# ============================================================

# IMPORTANTE:
# Los códigos geográficos de GDELT son FIPS, no ISO.
# Colombia CO | Bolivia BL | Costa Rica CS | Guatemala GT
# Venezuela VE | Brasil BR | Perú PE | México MX
# Chile CI | Uruguay UY | Ecuador EC

eventos = [
    ("Colombia",    "CO", "2018-06-17"),
    ("Bolivia",     "BL", "2008-09-11"),
    ("Costa Rica",  "CS", "2020-10-05"),
    ("Guatemala",   "GT", "2003-07-24"),
    ("Guatemala",   "GT", "2020-11-21"),
    ("Venezuela",   "VE", "2017-07-30"),
    ("Brasil",      "BR", "2013-06-17"),
    ("Perú",        "PE", "2016-06-05"),
    ("Perú",        "PE", "2020-11-10"),
    ("México",      "MX", "2023-02-26"),
    ("Chile",       "CI", "2023-03-08"),
    ("Uruguay",     "UY", "2023-03-08"),
    ("Venezuela",   "VE", "2004-06-03"),
    ("Brasil",      "BR", "2006-10-29"),
    ("Ecuador",     "EC", "2006-10-15"),
    ("Ecuador",     "EC", "2008-09-28"),
    ("Venezuela",   "VE", "2010-09-26"),
    ("Venezuela",   "VE", "2020-12-06")
]

df_eventos = pd.DataFrame(
    eventos,
    columns=["pais", "codigo", "fecha_evento"]
)

df_eventos["fecha_evento"] = pd.to_datetime(
    df_eventos["fecha_evento"]
)

# Ventanas de análisis
dias_pre = 30
dias_post = 7
dias_extra = 2

df_eventos["inicio_pre"] = (
    df_eventos["fecha_evento"] -
    pd.Timedelta(days=dias_pre)
)

df_eventos["fin_pre"] = (
    df_eventos["fecha_evento"] -
    pd.Timedelta(days=1)
)

df_eventos["inicio_post"] = (
    df_eventos["fecha_evento"] +
    pd.Timedelta(days=1)
)

df_eventos["fin_post"] = (
    df_eventos["fecha_evento"] +
    pd.Timedelta(days=dias_post)
)

# ============================================================
# 2. IDENTIFICAR LOS ARCHIVOS NECESARIOS
# ============================================================

fecha_inicio_diarios = pd.Timestamp("2013-04-01")
fecha_inicio_mensuales = pd.Timestamp("2006-01-01")

def clave_archivo(fecha):

    if fecha < fecha_inicio_mensuales:
        return ("anual", fecha.strftime("%Y"))

    elif fecha < fecha_inicio_diarios:
        return ("mensual", fecha.strftime("%Y%m"))

    else:
        return ("diario", fecha.strftime("%Y%m%d"))


archivos_eventos = {}
eventos_archivo = {}

for i, evento in df_eventos.iterrows():

    inicio = evento["inicio_pre"]
    fin = evento["fin_post"]

    # En los archivos diarios, los acontecimientos pueden
    # aparecer hasta algunos días después de su fecha efectiva.
    if evento["fecha_evento"] >= fecha_inicio_diarios:
        fin_archivo = fin + pd.Timedelta(days=dias_extra)
    else:
        fin_archivo = fin

    claves = set()

    for fecha in pd.date_range(inicio, fin_archivo, freq="D"):
        clave = clave_archivo(fecha)
        claves.add(clave)

        if clave not in eventos_archivo:
            eventos_archivo[clave] = set()

        eventos_archivo[clave].add(i)

    archivos_eventos[i] = claves

print("Archivos únicos a consultar:", len(eventos_archivo))
print("Archivos diarios:", sum(k[0] == "diario" for k in eventos_archivo))
print("Archivos mensuales:", sum(k[0] == "mensual" for k in eventos_archivo))
print("Archivos anuales:", sum(k[0] == "anual" for k in eventos_archivo))

# ============================================================
# 3. DESCARGA CON CACHÉ LOCAL
# ============================================================

carpeta_cache = Path("gdelt_cache")
carpeta_cache.mkdir(exist_ok=True)

session = requests.Session()

def nombre_archivo(clave):

    tipo, fecha = clave

    if tipo == "anual":
        return f"{fecha}.zip"

    elif tipo == "mensual":
        return f"{fecha}.zip"

    else:
        return f"{fecha}.export.CSV.zip"


def descargar_archivo(clave):

    nombre = nombre_archivo(clave)
    ruta = carpeta_cache / nombre

    # Reutilizar archivos ya descargados
    if ruta.exists():
        return ruta

    url = f"https://data.gdeltproject.org/events/{nombre}"

    for intento in range(5):

        response = session.get(url, timeout=180)

        if response.status_code == 404:
            print("Archivo no encontrado:", nombre)
            return None

        if response.status_code == 429 or response.status_code >= 500:
            time.sleep(2 ** intento)
            continue

        response.raise_for_status()

        ruta.write_bytes(response.content)

        # Pausa breve para no saturar el servidor
        time.sleep(0.1)

        return ruta

    print("No se pudo descargar:", nombre)
    return None

# ============================================================
# 4. LEER ARCHIVOS Y ASIGNAR REGISTROS A EVENTOS
# ============================================================

datos_eventos = {
    i: [] for i in range(len(df_eventos))
}

archivos_faltantes = set()

for contador, (clave, indices_eventos) in enumerate(
    eventos_archivo.items(), start=1
):

    print(
        f"\nArchivo {contador}/{len(eventos_archivo)}:",
        nombre_archivo(clave)
    )

    ruta = descargar_archivo(clave)

    if ruta is None:
        archivos_faltantes.add(clave)
        continue

    try:

        with zipfile.ZipFile(ruta) as z:

            miembros = [
                m for m in z.namelist()
                if not m.endswith("/")
            ]

            if not miembros:
                raise ValueError("ZIP sin archivos de datos")

            partes = []

            for miembro in miembros:

                parte = pd.read_csv(
                    z.open(miembro),
                    sep="\t",
                    header=None,
                    usecols=[0, 1, 29, 33, 51],
                    dtype={
                        0: str,
                        1: str,
                        51: str
                    },
                    low_memory=False
                )

                partes.append(parte)

            df_archivo = pd.concat(
                partes,
                ignore_index=True
            )

        df_archivo.columns = [
            "GlobalEventID",
            "SQLDATE",
            "QuadClass",
            "NumArticles",
            "ActionGeo_CountryCode"
        ]

        df_archivo["fecha"] = pd.to_datetime(
            df_archivo["SQLDATE"].astype(str).str.zfill(8),
            format="%Y%m%d",
            errors="coerce"
        )

        df_archivo["QuadClass"] = pd.to_numeric(
            df_archivo["QuadClass"],
            errors="coerce"
        )

        df_archivo["NumArticles"] = pd.to_numeric(
            df_archivo["NumArticles"],
            errors="coerce"
        ).fillna(0)

        # Asignar registros a cada evento correspondiente
        for i in indices_eventos:

            evento = df_eventos.loc[i]

            filtro = (
                (df_archivo["ActionGeo_CountryCode"] == evento["codigo"]) &
                (df_archivo["fecha"] >= evento["inicio_pre"]) &
                (df_archivo["fecha"] <= evento["fin_post"])
            )

            sub = df_archivo.loc[filtro].copy()

            if not sub.empty:
                datos_eventos[i].append(sub)

    except Exception as e:
        print("Error procesando", nombre_archivo(clave), ":", e)
        archivos_faltantes.add(clave)

# ============================================================
# 5. CALCULAR INDICADOR DIARIO Y PROMEDIOS
# ============================================================

resultados = []

for i, evento in df_eventos.iterrows():

    if datos_eventos[i]:

        df = pd.concat(
            datos_eventos[i],
            ignore_index=True
        )

        # Evitar duplicados
        df = df.drop_duplicates(
            subset="GlobalEventID"
        )

        df["articulos_conflicto"] = np.where(
            df["QuadClass"].isin([3, 4]),
            df["NumArticles"],
            0
        )

        diario = (
            df.groupby("fecha")
            .agg(
                articulos_totales=("NumArticles", "sum"),
                articulos_conflicto=("articulos_conflicto", "sum")
            )
        )

        diario["indicador"] = (
            100 *
            diario["articulos_conflicto"] /
            diario["articulos_totales"].replace(0, np.nan)
        )

    else:

        diario = pd.DataFrame(
            columns=[
                "articulos_totales",
                "articulos_conflicto",
                "indicador"
            ]
        )

    # Calendario completo: los días sin datos quedan como NaN
    calendario = pd.date_range(
        evento["inicio_pre"],
        evento["fin_post"],
        freq="D"
    )

    diario = diario.reindex(calendario)

    # Ventana basal
    pre = diario.loc[
        evento["inicio_pre"]:evento["fin_pre"],
        "indicador"
    ].dropna()

    # Ventana posterior
    post = diario.loc[
        evento["inicio_post"]:evento["fin_post"],
        "indicador"
    ].dropna()

    # Día del acontecimiento
    fecha = evento["fecha_evento"]

    if fecha in diario.index:
        indicador_evento = diario.loc[fecha, "indicador"]
    else:
        indicador_evento = np.nan

    # Archivos faltantes asociados a este evento
    n_archivos_faltantes = len(
        archivos_eventos[i] & archivos_faltantes
    )

    resultados.append({

        "pais": evento["pais"],
        "codigo": evento["codigo"],
        "fecha_evento": fecha,

        "promedio_pre": pre.mean(),
        "promedio_post": post.mean(),

        "indicador_evento": indicador_evento,

        "n_dias_pre": len(pre),
        "n_dias_post": len(post),

        "archivos_faltantes": n_archivos_faltantes

    })

# ============================================================
# 6. DATAFRAME FINAL
# ============================================================

df_resultados = pd.DataFrame(resultados)

df_resultados = df_resultados.sort_values(
    "fecha_evento"
).reset_index(drop=True)

# Redondear indicadores
columnas_indicadores = [
    "promedio_pre",
    "promedio_post",
    "indicador_evento"
]

df_resultados[columnas_indicadores] = (
    df_resultados[columnas_indicadores].round(2)
)

print("\n" + "=" * 80)
print("RESULTADOS GDELT")
print("=" * 80)

print(df_resultados.to_string(index=False))

print("\nArchivos faltantes:", len(archivos_faltantes))

Archivos únicos a consultar: 323
Archivos diarios: 312
Archivos mensuales: 9
Archivos anuales: 2

Archivo 1/323: 20180518.export.CSV.zip

Archivo 2/323: 20180519.export.CSV.zip

Archivo 3/323: 20180520.export.CSV.zip

Archivo 4/323: 20180521.export.CSV.zip

Archivo 5/323: 20180522.export.CSV.zip

Archivo 6/323: 20180523.export.CSV.zip

Archivo 7/323: 20180524.export.CSV.zip

Archivo 8/323: 20180525.export.CSV.zip

Archivo 9/323: 20180526.export.CSV.zip

Archivo 10/323: 20180527.export.CSV.zip

Archivo 11/323: 20180528.export.CSV.zip

Archivo 12/323: 20180529.export.CSV.zip

Archivo 13/323: 20180530.export.CSV.zip

Archivo 14/323: 20180531.export.CSV.zip

Archivo 15/323: 20180601.export.CSV.zip

Archivo 16/323: 20180602.export.CSV.zip

Archivo 17/323: 20180603.export.CSV.zip

Archivo 18/323: 20180604.export.CSV.zip

Archivo 19/323: 20180605.export.CSV.zip

Archivo 20/323: 20180606.export.CSV.zip

Archivo 21/323: 20180607.export.CSV.zip

Archivo 22/323: 20180608.export.CSV.zip

Archivo 2